# Réplica de Schneider et al. (2023) en Python

- **Artículo:** Schneider, K. R., Christiaensen, L., Webb, P. y Masters, W. A. (2023). *Assessing the affordability of nutrient-adequate diets*. American Journal of Agricultural Economics, 105(2), 503-524. https://doi.org/10.1111/ajae.12334
- **Datos:** Harvard Dataverse, https://doi.org/10.7910/DVN/AXDJZ8
- **Autores de la réplica:** Leandro Zegarra Garcia y Jesus Andre Aybar Reyes, Universidad de Lima, 2026-2.

## Paso 0. Descargar el código y los datos

In [ ]:
# Copia el repositorio de GitHub a este Colab y entra a la carpeta
!git clone -q https://github.com/Leozegarra/Replica_Schneider2023.git
%cd Replica_Schneider2023
# Colab ya trae numpy, pandas, scipy y matplotlib; esto solo confirma las versiones
!pip install -q -r requirements.txt
!ls

## Ejemplo: canasta de menor costo de una familia en un mes

$$\min \sum_j p_j \, q_j \quad \text{sujeto a: energía exacta, nutrientes} \ge \text{EAR}, \ \le \text{UL}, \ \text{macronutrientes dentro del AMDR}, \ q_j \ge 0$$

- $q_j$: gramos de cada alimento; $p_j$: su precio por gramo en el mercado de la familia ese mes.
- **EAR** es el mínimo de cada nutriente, **UL** el máximo que no hay que pasar y **AMDR** el rango de calorías que debe venir de carbohidratos, proteínas y grasas.
- Con la olla compartida, el mínimo de cada nutriente se fija con el miembro que más necesita por cada caloría.

Se compara con el costo calculado por la autora para la misma familia.

In [ ]:
# ---- Ejemplo: la canasta más barata de UNA familia en UN mes ----
import numpy as np, pandas as pd
from scipy.optimize import linprog

NUTRIENTES = ["energy", "carbohydrate", "protein", "lipids", "vitA", "retinol", "vitC", "vitE",
              "thiamin", "riboflavin", "niacin", "vitB6", "folate", "vitB12", "calcium", "copper",
              "iron", "magnesium", "phosphorus", "selenium", "zinc", "sodium"]
serie = pd.read_csv("datos/serie_mensual.csv.gz", low_memory=False, parse_dates=["dateMY"])
req = pd.read_csv("datos/requerimientos_hogar.csv.gz")
precios = pd.read_csv("datos/precios_mercados.csv.gz")

# 1) Elegimos una familia rural y un mes en que sí tuvo dieta compartida (puedes cambiar el número)
casos = serie[(serie.reside == 1) & (serie.data_round != 1) & serie.CoNA_sharing.notna()]
fila = casos.iloc[1000]
r = req[req.hhxround == fila.hhxround].iloc[0]
print(f"Familia {fila.hhxround}: {int(fila.hhsize)} personas, mercado {int(fila.market_no)}, {fila.dateMY:%m/%Y}")

# 2) Alimentos con precio ese mes en su mercado: precio por gramo y nutrientes por gramo
p = precios[(precios.market_no == fila.market_no) & (precios.year == fila.dateMY.year) & (precios.month == fila.dateMY.month)]
p = p[p.uni_price_net.notna() & (p.uni_price_net > 0)]
c = p.uni_price_net.values / 1000          # kwachas por gramo
A = p[NUTRIENTES].fillna(0).values.T / 100 # nutriente por gramo
print(f"Alimentos disponibles: {len(p)}")

# 3) Restricciones: energía exacta; mínimos (EAR, AMDR bajo) y máximos (UL, AMDR alto)
filas, lim = [], []
for k in range(1, 22):
    for nombre, signo in (("EAR", -1), ("AMDRlow", -1), ("UL", 1), ("AMDRup", 1)):
        v = r[f"{nombre}_sharing_{k + 2}"]
        if pd.notna(v) and v != 0:
            filas.append(signo * A[k]); lim.append(signo * v)
res = linprog(c, A_ub=np.array(filas), b_ub=np.array(lim), A_eq=A[:1], b_eq=[r["EAR_sharing_2"]],
              bounds=(0, None), method="highs")

# 4) Resultado: qué comprar, cuánto cuesta, y comparación con el costo que calculó la autora
canasta = pd.DataFrame({"alimento": p.food_item.values, "gramos_por_dia": res.x / 7,
                        "costo_por_dia_MWK": c * res.x / 7})
canasta = canasta[canasta.gramos_por_dia > 0.5].sort_values("gramos_por_dia", ascending=False)
print(canasta.round(1).to_string(index=False))
print(f"\nCosto de la familia: {res.fun / 7:,.0f} kwachas al día "
      f"= {res.fun / 7 / fila.PPP_month_interpolated / fila.hhsize:.2f} US$ PPP por persona")
print(f"Costo que calculó la autora: {fila.CoNA_sharing / 7:,.0f} kwachas al día")


## Paso 1. Olla compartida para todas las familias (unos 3 minutos)

Repite el ejemplo anterior para las 3117 familias, en cada mes de 2013 a 2017: **83 431 programas lineales**. Al final compara cada costo con el que publicó la autora.

In [ ]:
!python 01_resolver_canastas.py

## Paso 2. Tablas y figuras del artículo

Con los costos de cada familia se arman las tablas del paper. Todas las medias y medianas usan el **peso muestral** (cuántas familias reales representa cada familia encuestada).

- **Tabla 1:** quiénes son las familias (tamaño, gasto en comida).
- **Tabla 2:** en cuántos meses existe una dieta adecuada y cuánto cuesta.
- **Tabla 3:** si las familias pueden pagarla con lo que gastan hoy.

In [ ]:
!python 02_tablas_y_figuras.py > resultados/log_paso2.txt
import pandas as pd
from IPython.display import Image, display
for t in ["tabla1", "tabla2", "tabla3"]:
    print(f"\n{t.upper()}")
    display(pd.read_csv(f"resultados/{t}.csv", index_col=0))
for f in ["figura2_factibilidad", "figura3_costo", "figura5_acceso"]:
    display(Image(f"resultados/{f}.png", width=900))

## Paso 3. Dietas individuales con nuestro propio solver (unos 2 minutos)

Ahora cada persona come su propia dieta ideal: un programa lineal por grupo de edad y sexo, en cada mercado y mes (**40 325 problemas**). Después se suma el costo de los miembros de cada familia; si a uno solo le falta dieta, la familia no tiene dieta ese mes.

La salida compara nuestros costos con los de la autora. Sale casi igual; la diferencia viene de los niños de 1 a 3 años, cuyo costo publicado no se puede reproducir con los datos públicos.

In [ ]:
!python 04_dietas_individuales.py

## Paso 4. Estacionalidad (Tabla 2 y Figura 4)

¿En qué meses comer bien cuesta más o no se puede? Para cada familia se mide cuánto cambia el costo de un mes a otro y se estima, con una regresión ponderada, el efecto de cada mes del año. El **factor estacional** de un mes es cuánto más (o menos) cuesta que el promedio, y la **brecha estacional** es la diferencia entre el mes más caro y el más barato.

In [ ]:
!python 05_estacionalidad.py > resultados/log_paso5.txt
display(pd.read_csv("resultados/tabla2_estacionalidad.csv"))
display(pd.read_csv("resultados/brechas_estacionales.csv", index_col=0))
display(Image("resultados/figura4_estacionalidad.png", width=900))

## Paso 5. Comparación final con el artículo

Pone lado a lado los números del paper y los nuestros (24 indicadores).

In [ ]:
!python 03_comparar_con_paper.py > resultados/log_paso3.txt
pd.set_option("display.max_colwidth", 70)
display(pd.read_csv("resultados/comparacion_con_paper.csv"))